# CRUX Step 4: frozen splits

This notebook builds the train, dev_id, validation, and test splits and the evaluation-only benchmark files. It checks the leakage rules, compares the SHA-256 of each file with the frozen manifest, and shows the leakage report and the baselines. It does not train or evaluate a model. It does not print any test example.

In [ ]:
# Clone the repo and install the requirements.
!git clone https://github.com/0xAnto/crux.git /content/crux
%cd /content/crux
!pip install -q -r requirements.txt

The build reads the benchmark files of Step 2 and the synthetic data of Step 3. This cell makes them. Two generator runs give the same files (Step 3).

In [ ]:
!python -m crux.data.boardgameqa
!python -m crux.data.dynabench
!python -m crux.data.supernat
!python -m crux.gen.generator

The build downloads the pinned ProofWriter zip (SHA-256 check) for the test-only sets: depth 5, Birds-Electricity, and ParaRules. It checks the leakage rules and writes `data/splits/`. The build writes to a temporary folder first. It stops with an error if a file has another SHA-256 than `configs/splits_manifest.json`, and then the frozen files stay. Thus a clean run proves that the frozen splits are reproducible.

In [ ]:
!python -m crux.splits build
!python -m crux.splits verify

The leakage report: the overlap of the n-grams (lower is better) and the removed records.

In [ ]:
import json
m = json.load(open('configs/splits_manifest.json'))
print(m['leakage']['ngram']['n'], m['leakage']['ngram']['max_overlap'])
for pair, v in m['leakage']['ngram']['pairs'].items():
    print(f"{pair:22s} n={v['n']:>6d} max={v['max']:.4f} p99={v['p99']:.4f} over={v['over']}")
print('removed:', {k: len(v['ids']) for k, v in m['leakage']['removed'].items()})

The baselines (accuracy, higher is better). They are reference values. A model must score above them. They are for train, dev_id, validation, and the benchmarks. The test baselines are computed in Step 8.

In [ ]:
for split, parts in m['baselines']['majority'].items():
    for part, v in parts.items():
        print(f"{split:12s} {part:32s} n={v['n']:>6d} majority={v['majority']:.4f}")
print('fact-count baseline', m['baselines']['fact_count'])
print('edit pairs', json.dumps(m['baselines']['pairs']['validation']))
print('test:', {k: m['splits']['test'][k] for k in ('n', 'source', 'task_type', 'subset')})